# 🚀 DataMate Agent - Google Colab Training & Evaluation Pipeline
### Zalo AI Challenge: Vietnamese Offline Tabular QA System

Notebook này thiết lập môi trường GPU, tự động tải dữ liệu từ Google Drive, chạy vòng lặp **Iterative Self-Training (QLoRA + Rejection Sampling)** và đánh giá điểm số **Penalized Accuracy (PA)**.

## 1. Kiểm tra Phần cứng GPU (Khuyến nghị T4, A100 hoặc L4)

In [ ]:
!nvidia-smi

## 2. Clone Repository từ GitHub

In [ ]:
# Clone mã nguồn mới nhất từ repo của bạn
!rm -rf /content/datamate_zalo_ai
!git clone https://github.com/tuannh1905/datamate_zalo_ai.git /content/datamate_zalo_ai
%cd /content/datamate_zalo_ai/datamate_agent

## 3. Cài đặt Môi trường (vLLM, DuckDB, Unsloth, HuggingFace Base Weights)

In [ ]:
# Cài đặt dependencies và tải weights offline Qwen2.5-Coder-3B-Instruct
!bash scripts/setup_colab.sh

## 4. Tải và Chuẩn hóa Dữ liệu Cuộc thi (Khắc phục triệt để lỗi tải Google Drive)

> **Tại sao tải dataset Google Drive trên Colab hay bị lỗi?**
- Lệnh `gdown --folder` thường bị Google Drive giới hạn số lượt tải (rate-limit / virus scan warning) khi folder có hàng nghìn file CSV con.
- Đường dẫn bị lồng thư mục sai lệch (`train/train` so với `train/`) dẫn đến `FileNotFoundError`.

**Giải pháp:** Script `scripts/download_dataset.py` đã được tích hợp cơ chế tự động tìm kiếm, giải nén siêu tốc vào ổ NVMe cục bộ (`/content/`), tự động chuẩn hóa cấu trúc thư mục và chạy **Health Check** xác thực dữ liệu trước khi train.

### Cách 1 (Khuyên dùng - Ổn định 100%): Mount Google Drive cá nhân
Nếu bạn đã tải file zip hoặc lưu folder `train` / `public_test` trên Google Drive của mình, hãy chạy cell này:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tự động tìm kiếm file zip/folder trong Google Drive cá nhân, copy sang bộ nhớ NVMe và chuẩn hóa cấu trúc
!python scripts/download_dataset.py --mode auto

# Kiểm tra tính toàn vẹn của dữ liệu
!python scripts/download_dataset.py --mode check_only

### Cách 2: Tải tự động trực tiếp từ Google Drive Link qua gdown (Kèm auto-retry & chuẩn hóa)

In [ ]:
!pip install -q --upgrade gdown
!python scripts/download_dataset.py \
    --mode gdown_folder \
    --url "https://drive.google.com/drive/folders/1sMB8d47li018WyG_SsXIZJ02jfL8OaLi" \
    --target_base /content/dataset_gdrive

## 5. Chạy Thử Nghiệm Nhanh (Smoke Test trên 5 câu hỏi)

In [ ]:
# Chạy thử suy luận và chấm điểm nhanh 5 câu hỏi để xác nhận luồng chạy trơn tru
!python scripts/run_eval.py \
    --dataset train \
    --drive_base_path /content/dataset_gdrive \
    --run_inference \
    --limit 5 \
    --verbose

## 6. Chạy Vòng lặp Tự huấn luyện (Iterative Self-Training Loop)
Quy trình tự học 3 chu kỳ:
- **Vòng 1:** Model Base sinh 12 mẫu $\to$ Rejection Sampling chọn đáp án đúng $\to$ Huấn luyện LoRA Adapter `v1`.
- **Vòng 2:** Model + Adapter `v1` sinh mẫu $\to$ Rejection Sampling $\to$ Huấn luyện LoRA Adapter `v2`.
- **Vòng 3:** Model + Adapter `v2` sinh mẫu $\to$ Rejection Sampling $\to$ Huấn luyện Adapter `v3` hoàn chỉnh.

In [ ]:
!DRIVE_BASE_PATH="/content/dataset_gdrive" \
 DATASET="train" \
 bash scripts/run_self_training.sh

## 7. Đánh giá Toàn diện với Adapter v3 và Đo lường Penalized Accuracy

In [ ]:
!python scripts/run_eval.py \
    --dataset train \
    --drive_base_path /content/dataset_gdrive \
    --adapter ./weights/adapters/v3 \
    --run_inference

## 8. Sao lưu Adapter Weights vào Google Drive cá nhân

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ZaloAI_DataMate_Backup
!cp -r ./weights/adapters /content/drive/MyDrive/ZaloAI_DataMate_Backup/
print("✓ Đã sao lưu adapter weights thành công vào Google Drive!")